# DocRestore - Data Exploration

**Goal:** Visualise sample pairs and compute basic dataset statistics for both ShabbyPages and NoisyOffice.

Sections:
1. Configuration
2. Load 10 sample pairs from each dataset
3. Side-by-side visualisation (clean vs degraded)
4. Pixel intensity histograms
5. Dataset statistics

In [ ]:
# ── 0. Colab setup (skip if running locally) ─────────────────────────────
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/DocRestore'
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    sys.path.insert(0, PROJECT_ROOT)
    os.chdir(PROJECT_ROOT)

print('Project root:', PROJECT_ROOT)

In [ ]:
# -- 1. Imports & configuration ------------------------------------------------
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

random.seed(42)
np.random.seed(42)

# Dataset roots
SHABBY_CLEAN    = Path('data/shabby/clean')
SHABBY_DEGRADED = Path('data/shabby/degraded')
NOISY_CLEAN     = Path('data/noisy/clean')
NOISY_DEGRADED  = Path('data/noisy/degraded')

N_SAMPLES = 10

IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}

# Report availability
for name, p in [('ShabbyPages clean', SHABBY_CLEAN),
                ('ShabbyPages degraded', SHABBY_DEGRADED),
                ('NoisyOffice clean', NOISY_CLEAN),
                ('NoisyOffice degraded', NOISY_DEGRADED)]:
    status = 'OK' if p.exists() else 'MISSING'
    print(f'  [{status}] {name}: {p}')

In [ ]:
# ── 2. Helper: load matched pairs ─────────────────────────────────────────

def _pair_key(stem: str) -> str:
    """
    Normalise a filename stem to a pairing key that works for both datasets.

    ShabbyPages: clean and degraded share the exact same stem.
      key = stem  (unchanged)

    NoisyOffice: stems differ by the middle token.
      clean    Fontfre_Clean_TE  → key  Fontfre__TE
      degraded Fontfre_Noisec_TE → key  Fontfre__TE
    """
    parts = stem.split('_')
    if len(parts) >= 3 and parts[1].lower().startswith(('clean', 'noise')):
        # NoisyOffice convention: <font>_<type>_<split>
        return f'{parts[0]}__{parts[-1]}'
    return stem  # ShabbyPages or any other dataset


def load_pairs(clean_dir: Path, degraded_dir: Path, n: int = 10):
    """Return a list of (clean_PIL, degraded_PIL, label) for *n* random pairs."""
    if not clean_dir.exists() or not degraded_dir.exists():
        print(f'  SKIPPED — directory not found: {clean_dir} / {degraded_dir}')
        return []

    clean_files    = {p: _pair_key(p.stem)
                      for p in clean_dir.iterdir()
                      if p.suffix.lower() in IMAGE_EXTS}
    degraded_files = {p: _pair_key(p.stem)
                      for p in degraded_dir.iterdir()
                      if p.suffix.lower() in IMAGE_EXTS}

    # Invert: key → first matching path
    clean_by_key    = {v: k for k, v in clean_files.items()}
    degraded_by_key = {v: k for k, v in degraded_files.items()}

    matched_keys = sorted(clean_by_key.keys() & degraded_by_key.keys())
    print(f'  Matched pairs found: {len(matched_keys)}')

    chosen = random.sample(matched_keys, min(n, len(matched_keys)))
    pairs = []
    for key in chosen:
        c = Image.open(clean_by_key[key]).convert('RGB')
        d = Image.open(degraded_by_key[key]).convert('RGB')
        pairs.append((c, d, key))
    return pairs


print('Loading ShabbyPages pairs...')
shabby_pairs = load_pairs(SHABBY_CLEAN, SHABBY_DEGRADED, N_SAMPLES)

print('Loading NoisyOffice pairs...')
noisy_pairs  = load_pairs(NOISY_CLEAN,  NOISY_DEGRADED,  N_SAMPLES)

In [ ]:
# -- 3. Side-by-side visualisation --------------------------------------------

def show_pairs(pairs, dataset_name: str, thumb_size=(256, 256)):
    if not pairs:
        print(f'  [{dataset_name}] No pairs to display - skipping.')
        return
    n = len(pairs)
    fig, axes = plt.subplots(2, n, figsize=(n * 2.5, 6))
    fig.suptitle(f'{dataset_name}  -  Top row: Clean  |  Bottom row: Degraded',
                 fontsize=13, fontweight='bold')

    for col, (clean, deg, label) in enumerate(pairs):
        c_thumb = clean.resize(thumb_size, Image.LANCZOS)
        d_thumb = deg.resize(thumb_size, Image.LANCZOS)

        axes[0, col].imshow(c_thumb)
        axes[0, col].set_title(str(label)[:14], fontsize=7)
        axes[0, col].axis('off')

        axes[1, col].imshow(d_thumb)
        axes[1, col].axis('off')

    plt.tight_layout()
    plt.show()


show_pairs(shabby_pairs, 'ShabbyPages')
show_pairs(noisy_pairs,  'NoisyOffice')

In [ ]:
# -- 4. Pixel intensity histograms --------------------------------------------

def pixel_histogram(pairs, dataset_name: str):
    if not pairs:
        print(f'  [{dataset_name}] No pairs - skipping histogram.')
        return

    clean_pixels    = np.concatenate(
        [np.array(c).ravel() / 255.0 for c, _, _ in pairs])
    degraded_pixels = np.concatenate(
        [np.array(d).ravel() / 255.0 for _, d, _ in pairs])

    fig, ax = plt.subplots(figsize=(8, 4))
    bins = np.linspace(0, 1, 64)
    ax.hist(clean_pixels,    bins=bins, alpha=0.6, color='steelblue',
            label='Clean',    density=True)
    ax.hist(degraded_pixels, bins=bins, alpha=0.6, color='tomato',
            label='Degraded', density=True)
    ax.set_title(f'{dataset_name} - Pixel Intensity Distribution (normalised)',
                 fontsize=12)
    ax.set_xlabel('Pixel value [0, 1]')
    ax.set_ylabel('Density')
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f'  {dataset_name} clean    - mean: {clean_pixels.mean():.3f}, '
          f'std: {clean_pixels.std():.3f}')
    print(f'  {dataset_name} degraded - mean: {degraded_pixels.mean():.3f}, '
          f'std: {degraded_pixels.std():.3f}')


pixel_histogram(shabby_pairs, 'ShabbyPages')
pixel_histogram(noisy_pairs,  'NoisyOffice')

In [ ]:
# -- 5. Dataset statistics -----------------------------------------------------

def dataset_stats(clean_dir: Path, degraded_dir: Path, dataset_name: str):
    if not clean_dir.exists() or not degraded_dir.exists():
        print(f'  [{dataset_name}] SKIPPED - directory not found.')
        return

    clean_files    = [p for p in clean_dir.iterdir()
                      if p.suffix.lower() in IMAGE_EXTS]
    degraded_files = [p for p in degraded_dir.iterdir()
                      if p.suffix.lower() in IMAGE_EXTS]

    def _stats(files):
        sizes, modes = [], set()
        for p in files:
            with Image.open(p) as img:
                sizes.append(img.size)
                modes.add(img.mode)
        widths  = [s[0] for s in sizes]
        heights = [s[1] for s in sizes]
        return dict(
            count=len(files),
            modes=modes,
            width_range=(min(widths), max(widths)),
            height_range=(min(heights), max(heights)),
            width_mean=float(np.mean(widths)),
            height_mean=float(np.mean(heights)),
        )

    cs = _stats(clean_files)
    ds = _stats(degraded_files)

    clean_keys    = {_pair_key(p.stem) for p in clean_files}
    degraded_keys = {_pair_key(p.stem) for p in degraded_files}
    matched = len(clean_keys & degraded_keys)

    sep = '-' * 50
    print(f'\n{sep}')
    print(f'  Dataset        : {dataset_name}')
    print(f'  Clean images   : {cs["count"]}')
    print(f'  Degraded images: {ds["count"]}')
    print(f'  Matched pairs  : {matched}')
    print(f'  Clean   sizes  : W {cs["width_range"]}, H {cs["height_range"]} '
          f'(mean {cs["width_mean"]:.0f}x{cs["height_mean"]:.0f})')
    print(f'  Degraded sizes : W {ds["width_range"]}, H {ds["height_range"]} '
          f'(mean {ds["width_mean"]:.0f}x{ds["height_mean"]:.0f})')
    print(f'  Clean   modes  : {cs["modes"]}')
    print(f'  Degraded modes : {ds["modes"]}')
    print(sep)


dataset_stats(SHABBY_CLEAN, SHABBY_DEGRADED, 'ShabbyPages')
dataset_stats(NOISY_CLEAN,  NOISY_DEGRADED,  'NoisyOffice')